# Marketing AI GPU server: Mistral 7B + Stable Diffusion 1.5

This notebook fine-tunes both models and runs the model server that your local backend calls.

**Before the first run**
1. **Runtime → Change runtime type → T4 GPU** (or any GPU).
2. Upload the whole `marketing-ai-local` project folder to Google Drive as `MyDrive/marketing-ai-local`.
   Trained adapters are saved there too, so you only train once.
3. Accept the model terms at https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 (log in, click *Agree*).
4. Add Colab secrets (key icon on the left): `HF_TOKEN` = your Hugging Face token (read access).
   Optional: `NGROK_AUTHTOKEN` for a stable ngrok tunnel instead of Cloudflare.

**Every time:** Runtime → Run all, then copy the two lines from the last cell into `backend/.env` and restart the backend.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Project folder and settings

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/marketing-ai-local'  # change if you put it somewhere else

# Training switches. Training runs only when the adapter does not exist yet,
# unless you set RETRAIN_* to True (for example after adding new examples or photos).
TRAIN_TEXT = True
RETRAIN_TEXT = False
TRAIN_IMAGE = True
RETRAIN_IMAGE = False
IMAGE_TRIGGER = 'brandstyle'   # the word your image LoRA learns your style under
TEXT_EPOCHS = 3

assert os.path.isdir(PROJECT), f'Project not found at {PROJECT}. Upload the marketing-ai-local folder to Google Drive.'
os.chdir(PROJECT)
print('Project:', PROJECT)

In [ ]:
!pip install -q -r model_server/requirements.txt

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

assert HF_TOKEN, 'Add your Hugging Face token as the Colab secret HF_TOKEN (key icon on the left).'
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

## 2. Fine-tune Mistral 7B (QLoRA)

Uses `training/data/seed_examples.jsonl`, plus `training/data/db_export.jsonl` and
`training/data/custom/*.jsonl` when present. About 15 to 40 minutes on a T4.

In [ ]:
TEXT_ADAPTER = 'training/output/mistral-marketing-lora'

if TRAIN_TEXT and (RETRAIN_TEXT or not os.path.exists(f'{TEXT_ADAPTER}/adapter_config.json')):
    !python training/build_dataset.py
    !python training/train_mistral_lora.py --epochs {TEXT_EPOCHS}
else:
    print('Skipping text training. Adapter:', TEXT_ADAPTER if os.path.exists(f'{TEXT_ADAPTER}/adapter_config.json') else 'none (base model)')

## 3. Fine-tune Stable Diffusion 1.5 (LoRA)

Put 15 to 60 of your product or brand photos in `training/data/brand_images/` on Drive
(optional captions in a `.txt` file with the same name). Skipped when there are fewer than 5 photos.

In [ ]:
import glob

IMAGE_LORA = 'training/output/sd15-brand-lora'
photos = [p for p in glob.glob('training/data/brand_images/*') if p.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))]
has_lora = bool(glob.glob(f'{IMAGE_LORA}/*.safetensors'))

if TRAIN_IMAGE and len(photos) >= 5 and (RETRAIN_IMAGE or not has_lora):
    !python training/train_sd_lora.py --trigger "{IMAGE_TRIGGER}"
elif TRAIN_IMAGE and len(photos) < 5:
    print(f'Found {len(photos)} photos: add at least 5 to training/data/brand_images to train the image LoRA. Using base SD 1.5.')
else:
    print('Skipping image training. LoRA:', IMAGE_LORA if has_lora else 'none (base model)')

## 4. Start the model server

In [ ]:
import secrets, subprocess, time, requests

# The key is kept on Drive, so backend/.env does not need a new key every session.
key_file = 'model_server/.api_key'
if not os.path.exists(key_file):
    with open(key_file, 'w') as f:
        f.write(secrets.token_urlsafe(32))
API_KEY = open(key_file).read().strip()

env = dict(os.environ, MODEL_SERVER_API_KEY=API_KEY, HF_TOKEN=HF_TOKEN, PYTHONUNBUFFERED='1')
subprocess.run('pkill -f "uvicorn server:app"', shell=True)
server_log = open('/content/model_server.log', 'w')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '7860'],
    cwd=f'{PROJECT}/model_server', env=env, stdout=server_log, stderr=subprocess.STDOUT,
)

for _ in range(180):
    time.sleep(5)
    try:
        health = requests.get('http://127.0.0.1:7860/health', timeout=5).json()
    except Exception:
        continue
    print(health['status'], '-', health['step'])
    if health['status'] in ('ready', 'error'):
        break

print(health)
if health.get('status') != 'ready':
    print(open('/content/model_server.log').read()[-3000:])

## 5. Public URL for your backend

In [ ]:
import re

try:
    NGROK = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    NGROK = None

if NGROK:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK)
    PUBLIC_URL = ngrok.connect(7860, 'http').public_url
else:
    if not os.path.exists('/content/cloudflared'):
        !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x /content/cloudflared
    subprocess.run('pkill -f "cloudflared tunnel"', shell=True)
    tunnel_log = open('/content/tunnel.log', 'w')
    tunnel = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
        stdout=tunnel_log, stderr=subprocess.STDOUT,
    )
    PUBLIC_URL = None
    for _ in range(60):
        time.sleep(1)
        found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
        if found:
            PUBLIC_URL = found.group(0)
            break

assert PUBLIC_URL, 'The tunnel did not start. Run this cell again.'

print('Paste these lines into backend/.env and restart the backend:')
print()
print(f'MODEL_SERVER_URL={PUBLIC_URL}')
print(f'MODEL_SERVER_API_KEY={API_KEY}')

## 6. Quick test

In [ ]:
import base64
from IPython.display import Image, display

headers = {'X-API-Key': API_KEY}
started = time.time()
reply = requests.post(f'{PUBLIC_URL}/v1/chat', headers=headers, timeout=120, json={
    'messages': [{'role': 'user', 'content': 'Weekend offer: buy 2 cupcakes, get 1 free'}],
    'max_new_tokens': 300,
}).json()
print(reply.get('text', reply))
print(f'{time.time() - started:.1f}s')

picture = requests.post(f'{PUBLIC_URL}/v1/image', headers=headers, timeout=120, json={
    'prompt': 'assorted cupcakes with creamy frosting on a rustic wooden table, cosy bakery, soft morning light, professional food photography',
}).json()
if 'image_base64' in picture:
    display(Image(data=base64.b64decode(picture['image_base64']), width=384))
    print(f"{picture['seconds']}s")
else:
    # The server answered with an error instead of an image.
    print('Image failed:', picture.get('detail', picture))
    print(open('/content/model_server.log').read()[-3000:])

Keep this tab open while you use the app. Free Colab sessions stop after a period of inactivity
or about 12 hours; run all cells again to restart (training is skipped because the adapters are on Drive).
With a Cloudflare tunnel the URL changes each session, so update `MODEL_SERVER_URL` in `backend/.env`.